# 03 — Reinforcement Learning: Simulasi dan Evaluasi Agent

Environment: FrozenLake-v1, Taxi-v3, CartPole-v1.

Reinforcement Learning tidak dievaluasi memakai accuracy/precision/recall seperti klasifikasi. Evaluasi menggunakan total reward per episode, rata-rata reward, panjang episode, dan success rate yang sesuai dengan definisi environment.

Notebook ini menjalankan **baseline random policy** (aksi dipilih acak). Ini belum merupakan agent yang sudah belajar.


## 1. Instalasi dan import

In [ ]:
!pip -q install pandas numpy "gymnasium[classic-control]"


In [ ]:
import gymnasium as gym
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display


## 2. Fungsi pengumpulan pengalaman dan evaluasi

In [ ]:
def evaluate_random_policy(env_id, episodes=100, seed=42, max_steps=2000):
    env = gym.make(env_id)
    transitions, episode_rows = [], []

    for ep in range(episodes):
        state, info = env.reset(seed=seed + ep)
        terminated = truncated = False
        step, total_reward = 0, 0.0

        while not (terminated or truncated) and step < max_steps:
            action = env.action_space.sample()
            next_state, reward, terminated, truncated, info = env.step(action)
            done = bool(terminated or truncated)

            transitions.append({
                "environment": env_id,
                "episode": ep + 1,
                "step": step + 1,
                "state": np.asarray(state).tolist() if isinstance(state, (np.ndarray, list, tuple)) else state,
                "action": int(action),
                "reward": float(reward),
                "next_state": np.asarray(next_state).tolist() if isinstance(next_state, (np.ndarray, list, tuple)) else next_state,
                "terminated": bool(terminated),
                "truncated": bool(truncated),
                "done": done
            })
            state = next_state
            total_reward += float(reward)
            step += 1

        episode_rows.append({
            "environment": env_id,
            "episode": ep + 1,
            "episode_length": step,
            "total_reward": total_reward,
            "terminated": bool(terminated),
            "truncated": bool(truncated)
        })

    env.close()
    return pd.DataFrame(transitions), pd.DataFrame(episode_rows)


## 3. FrozenLake-v1

Dokumentasi: [Gymnasium FrozenLake](https://gymnasium.farama.org/environments/toy_text/frozen_lake/). Default 4×4 memiliki 16 state diskret dan 4 aksi. Reward standar +1 saat mencapai goal, 0 untuk langkah lain atau masuk hole.


In [ ]:
frozen_transitions, frozen_episodes = evaluate_random_policy("FrozenLake-v1", episodes=100)
frozen_episodes["success"] = frozen_episodes["total_reward"] > 0
display(frozen_transitions.head())
display(frozen_episodes.head())
print("Mean return:", frozen_episodes["total_reward"].mean())
print("Mean episode length:", frozen_episodes["episode_length"].mean())
print("Goal success rate:", frozen_episodes["success"].mean())


## 4. Taxi-v3

Dokumentasi: [Gymnasium Taxi](https://gymnasium.farama.org/environments/toy_text/taxi/). State mengodekan posisi taxi, lokasi penumpang, dan tujuan; tersedia 500 state diskret dan 6 aksi. Episode berhasil ketika taxi menurunkan penumpang di lokasi tujuan (terminated).


In [ ]:
taxi_transitions, taxi_episodes = evaluate_random_policy("Taxi-v3", episodes=100)
taxi_episodes["success"] = taxi_episodes["terminated"]
display(taxi_transitions.head())
display(taxi_episodes.head())
print("Mean return:", taxi_episodes["total_reward"].mean())
print("Mean episode length:", taxi_episodes["episode_length"].mean())
print("Successful drop-off rate:", taxi_episodes["success"].mean())


## 5. CartPole-v1

Dokumentasi: [Gymnasium CartPole](https://gymnasium.farama.org/environments/classic_control/cart_pole/). State terdiri dari empat nilai kontinu: posisi cart, kecepatan cart, sudut pole, dan kecepatan sudut pole. Tersedia dua aksi (dorong kiri/kanan). Reward standar +1 untuk setiap langkah bertahan.

Untuk CartPole, laporkan rata-rata return dan panjang episode. Mencapai batas waktu 500 langkah (`truncated`) dicatat sebagai pencapaian batas episode, bukan label keberhasilan klasifikasi.


In [ ]:
cartpole_transitions, cartpole_episodes = evaluate_random_policy("CartPole-v1", episodes=100)
cartpole_episodes["reached_500_steps"] = cartpole_episodes["episode_length"] >= 500
display(cartpole_transitions.head())
display(cartpole_episodes.head())
print("Mean return:", cartpole_episodes["total_reward"].mean())
print("Mean episode length:", cartpole_episodes["episode_length"].mean())
print("Rate reaching 500-step limit:", cartpole_episodes["reached_500_steps"].mean())


## 6. Ringkasan evaluasi baseline random policy

In [ ]:
rl_summary = pd.DataFrame([
    {
        "environment": "FrozenLake-v1", "episodes": len(frozen_episodes),
        "mean_return": frozen_episodes["total_reward"].mean(),
        "std_return": frozen_episodes["total_reward"].std(),
        "mean_episode_length": frozen_episodes["episode_length"].mean(),
        "success_or_limit_rate": frozen_episodes["success"].mean(),
        "rate_definition": "goal success"
    },
    {
        "environment": "Taxi-v3", "episodes": len(taxi_episodes),
        "mean_return": taxi_episodes["total_reward"].mean(),
        "std_return": taxi_episodes["total_reward"].std(),
        "mean_episode_length": taxi_episodes["episode_length"].mean(),
        "success_or_limit_rate": taxi_episodes["success"].mean(),
        "rate_definition": "successful passenger drop-off"
    },
    {
        "environment": "CartPole-v1", "episodes": len(cartpole_episodes),
        "mean_return": cartpole_episodes["total_reward"].mean(),
        "std_return": cartpole_episodes["total_reward"].std(),
        "mean_episode_length": cartpole_episodes["episode_length"].mean(),
        "success_or_limit_rate": cartpole_episodes["reached_500_steps"].mean(),
        "rate_definition": "episodes reaching 500-step limit"
    }
])
display(rl_summary)


In [ ]:
plt.figure(figsize=(8, 4))
for name, data in [
    ("FrozenLake", frozen_episodes),
    ("Taxi-v3", taxi_episodes),
    ("CartPole", cartpole_episodes)
]:
    plt.plot(data["episode"], data["total_reward"], label=name)
plt.xlabel("Episode")
plt.ylabel("Total reward")
plt.title("Random Policy — Reward per Episode")
plt.legend()
plt.grid(True)
plt.show()


## 7. Ekspor pengalaman ke CSV

In [ ]:
frozen_transitions.to_csv("FrozenLake_experience.csv", index=False)
taxi_transitions.to_csv("Taxi_experience.csv", index=False)
cartpole_transitions.to_csv("CartPole_experience.csv", index=False)
rl_summary.to_csv("Reinforcement_evaluation_summary.csv", index=False)
print("File CSV berhasil dibuat.")


In [ ]:
from google.colab import files
for filename in [
    "FrozenLake_experience.csv", "Taxi_experience.csv",
    "CartPole_experience.csv", "Reinforcement_evaluation_summary.csv"
]:
    files.download(filename)


## 8. Referensi

1. Farama Foundation. *Gymnasium: FrozenLake*. https://gymnasium.farama.org/environments/toy_text/frozen_lake/
2. Farama Foundation. *Gymnasium: Taxi*. https://gymnasium.farama.org/environments/toy_text/taxi/
3. Farama Foundation. *Gymnasium: CartPole*. https://gymnasium.farama.org/environments/classic_control/cart_pole/
4. Sutton, R. S., & Barto, A. G. (2018). *Reinforcement Learning: An Introduction* (2nd ed.). MIT Press. http://incompleteideas.net/book/the-book-2nd.html

Baseline ini memilih aksi acak. Untuk menyatakan agent telah belajar, tambahkan algoritma seperti Q-learning pada state diskret atau metode yang sesuai untuk state kontinu, kemudian bandingkan hasilnya dengan baseline.
